# DM-NeRF / DM-SR — AMP Long Run Start (Kaggle)

Primeira sessão longa do experimento final. Ative GPU + Internet e use **Save Version → Save & Run All**. Pode fechar o navegador e desligar o computador depois que a versão aparecer como Running.

Código de treino fixado em um commit auditado. Esta sessão começa do zero para não misturar o checkpoint do benchmark AMP com os controles finais de GradScaler/effective updates.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

def run(*args):
    subprocess.run([str(x) for x in args],check=True)

PIN='e8bef9b7dc7732719c9ded4f0e1024db242c8baa'
run('nvidia-smi')
BASE=Path('/kaggle/temp/dissertacao-dmnerf-final')
OUTPUT=Path('/kaggle/working/dmnerf-results')
if BASE.exists(): shutil.rmtree(BASE)
if OUTPUT.exists(): shutil.rmtree(OUTPUT)
BASE.mkdir(parents=True,exist_ok=True)
REPO=BASE/'sdf-generator'; WORK=BASE/'work'
run('git','clone','https://github.com/Jfsslemos/sdf-generator.git',REPO)
run('git','-C',REPO,'checkout','--detach',PIN)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==PIN
run(sys.executable,REPO/'tools/dmnerf/bootstrap.py','--work',WORK)
PYTHON=WORK/'env/bin/python'
run(PYTHON,'-c','import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0),torch.__version__)')
run(PYTHON,REPO/'tools/dmnerf/prepare.py','--work',WORK)


In [ ]:
run(PYTHON,REPO/'tools/dmnerf/run.py','--work',WORK,'--output',OUTPUT,'--profile','full','--amp')


In [ ]:
state=json.loads((OUTPUT/'raw/study/experiment/training_state.json').read_text())
print(json.dumps(state,indent=2))
print('Iterações restantes:',max(0,state['target_steps']-(state['iteration']+1)))
print('Pacote:',OUTPUT.parent/(OUTPUT.name+'-bundle.zip'))


Quando terminar, use o output desta versão como **Add Input → Notebook Output** em `DMNeRF_Kaggle_AMP_Resume.ipynb`. Envie também `dmnerf-results-bundle.zip` ao ChatGPT para auditoria.
